# Reproducing Peterson et al. (2021) from DANDI:000055

Paper: Peterson, Singh, Wang, Rao and Brunton (2021), "Behavioral and Neural Variability of
Naturalistic Arm Movements", eNeuro 8(3), ENEURO.0007-21.2021, doi:10.1523/eneuro.0007-21.2021.

Dataset: DANDI:000055 (AJILE12), version 0.261123.1703 (doi:10.48324/dandi.000055/0.261123.1703).
This version was asked for in the prompt of this attempt, a repeat attempt that tests what the
dataset holds now. It holds 55 NWB files, one per subject and recording day, with ECoG at 500 Hz,
pose at 30 Hz, the contralateral wrist movement events and their features. Every file is streamed;
nothing is downloaded.

Claims: `claims/peterson-2021.yaml`, written from the version of record (Europe PMC, PMC8225404)
and checked by a second reader. Method details the claims file does not give are taken from the
paper's Methods and its code (github.com/BruntonUWBio/naturalistic_arm_movements_ecog, commit
04c6faf2f219586fa834d10c614ef48a694cec55), cited where they are used. Outside input: the paper's
projection matrices from electrodes to regions, Figshare item 12114408 version 1
(doi:10.6084/m9.figshare.12114408.v1).

The notebook runs for about 40 minutes: about 12 minutes to stream the ECoG around 8088 events
(about 150 GB of compressed chunks) and about 25 minutes for the 3904 regression models.

| Claim | Tested | Outcome |
|---|---|---|
| subject-days | yes | reproduced |
| roi-projection-link | yes (with the Figshare matrices) | reproduced |
| cohort | yes | partially reproduced |
| fig2-event-counts | yes | not reproduced |
| fig2-features-differ-across-subjects | yes | reproduced |
| fig2-reach-shape | yes | partially reproduced |
| fig2-bimanual-skew | yes | reproduced |
| extfig2-1c-contralateral-larger | yes | reproduced |
| extfig2-2-feature-correlations | yes | partially reproduced |
| fig2-days-and-waking | yes | partially reproduced |
| fig3-group-lfb-decrease-hfb-increase | yes | reproduced |
| fig3-sensorimotor-largest | yes | reproduced |
| fig3-lfb-more-widespread | yes | blocked (methods underspecified: the readings disagree) |
| fig4-precentral-subject-variability | yes | partially reproduced |
| fig5-band-power-differs-across-subjects | yes | reproduced |
| table1-subject-band-power-ci | yes | partially reproduced |
| fig5-recording-day-effects | yes | partially reproduced |
| extfig5-1-baseline-reduces-day-variability | yes | reproduced |
| discussion-day-changes-small | yes | reproduced |
| fig6b-best-models-sensorimotor | yes | reproduced |
| fig6-r2-at-most-025 | yes | reproduced |
| fig6c-angle-and-day-most-informative | yes | reproduced |
| extfig6-2-angle-most-retained | yes | partially reproduced |
| fig6d-upward-reaches-enhance-pattern | yes | reproduced |
| fig6d-angle-and-day-largest-coefficients | yes | reproduced |

Assumptions (the paper leaves these open; each is stated where it is used): how the segment
rejection pools power over electrodes and frequencies; that each event uses the good-electrode flags
of its own file for the region projection, and that the regression and electrode-level analyses use
the electrodes good in every file of the subject.

In [ ]:
import os, re, time, zlib, resource
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import ajile

KEY = "peterson-2021"
def seed(claim):
    """Seed of a claim's generator (protocol, author step 3)."""
    return zlib.crc32(f"{KEY}/{claim}".encode())
SEED_OFFSETS = [0, 1000, 2000, 3000, 4000]
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
T0 = time.time()
print("dandiset", ajile.DANDISET, "version", ajile.VERSION)

In [ ]:
# LOAD-META: list the assets of the pinned version and read every file's small datasets (streamed)
assets = ajile.list_assets()
metas = ajile.load_all_meta(assets)
print(len(assets), "files,", assets["size"].sum() / 1e9, "GB in all")

In [ ]:
ev_all = ajile.event_table(metas)
subjects = sorted(ev_all.subject.unique())
print("subjects:", subjects)
days = ev_all.groupby("subject").day.unique().map(sorted)
print("days per subject:", days.to_dict())
for m in metas[:1]:
    print("session_description:", m["session_description"])
# Per-file facts
rows = []
for m in metas:
    el = m["electrodes"]
    rows.append(dict(subject=int(m["subject_id"]), day=int(m["session_id"]), sex=m["sex"], age=m["age"],
                     start=m["session_start"][:10], ecog_hours=m["ecog_shape"][0] / m["ecog_rate"] / 3600,
                     n_el=m["ecog_shape"][1], n_el_table=len(el), good=int(el.good.sum()),
                     no_position=int(el.x.isna().sum()), events=len(m["reaches"]), event_wrist=m["event_wrist"],
                     pose_hours=m["pose_shape"][0] / m["pose_rate"] / 3600,
                     events_after_ecog_end=int((m["event_times"] + 5 > m["ecog_shape"][0] / m["ecog_rate"]).sum())))
files = pd.DataFrame(rows)
print(files.to_string())

In [ ]:
# Checks of the description against the values
m0 = metas[0]
print("ECoG unit:", m0["ecog_unit"], "conversion:", m0["ecog_conversion"], "rate:", m0["ecog_rate"], "start:", m0["ecog_start"],
      "chunks:", m0["ecog_chunks"])
print("pose unit:", m0["pose_unit"], "rate:", m0["pose_rate"], "series:", m0["pose_series"])
print("all ECoG at 500 Hz:", all(m["ecog_rate"] == 500 for m in metas), "; all pose at 30 Hz:", all(m["pose_rate"] == 30 for m in metas))
print("ElectricalSeries rows = electrodes table:", all(m["ecog_shape"][1] == len(m["electrodes"]) and
      np.array_equal(m["es_electrodes"], np.arange(len(m["electrodes"]))) for m in metas))
print("ReachEvents timestamps = reaches start_time (max diff, s):",
      max(np.abs(m["event_times"] - m["reaches"].start_time.values).max() for m in metas))
# electrode positions and flags across a subject's files
for s in subjects:
    ms = [m for m in metas if int(m["subject_id"]) == s]
    xyz = [m["electrodes"][["x", "y", "z"]].values for m in ms]
    same = all(np.array_equal(a, xyz[0], equal_nan=True) for a in xyz)
    G = np.array([m["electrodes"].good.values for m in ms])
    print(f"S{s:02d}: positions identical across files {same}; electrodes without position {int(np.isnan(xyz[0][:, 0]).sum())}; "
          f"bad per file {list((~G).sum(1))}; bad in every file {(~G).all(0).sum()}; bad in any file {(~G).any(0).sum()}")
print("columns of the reaches table:", list(metas[0]["reaches"].columns))
print("columns of the electrodes table:", list(metas[0]["electrodes"].columns))
print("columns of the epochs table:", list(metas[0]["epochs"].columns))
desc = ev_all[["duration", "magnitude", "angle_raw", "onset_speed", "speech_ratio", "Bimanual_ratio", "bimanual_overlap",
               "bimanual_class", "time_of_day"]].describe().T
print(desc)
print("missing values per reaches column:", ev_all[["Reach_magnitude_px", "Reach_angle_degrees", "Onset_speed_px_per_sec",
      "Speech_ratio", "Bimanual_ratio", "Bimanual_overlap", "Bimanual_class"]].isna().sum().to_dict())
print("bimanual_class values:", sorted(ev_all.bimanual_class.unique()))

In [ ]:
# Coarse behavioural labels (epochs) at the event times. The Methods exclude movements during
# sleep, unrelated experiments and private times.
lab = []
for m in metas:
    ep = m["epochs"]
    for t in m["event_times"]:
        k = np.flatnonzero((ep.start_time.values <= t) & (ep.stop_time.values > t))
        lab.append(ep.labels.values[k[0]] if len(k) else "(none)")
ev_all["epoch_label"] = lab
print(ev_all.epoch_label.value_counts().to_string())

## Projection matrices (outside input) and claim `roi-projection-link`

> The rows of the paper's projection matrices from electrodes to the eight regions, deposited on
> Figshare, can be matched one to one to the dataset's electrodes, and their weights fit the
> electrodes' positions.

Figshare item 12114408, version 1 (doi:10.6084/m9.figshare.12114408.v1), read at run time. Rows are
matched as the paper's code does (`tfr_utils.load_project_spectral_power`, commit 04c6faf): it keeps
the non-zero rows and drops the first of them (`np.nonzero(df.transpose().mean().values != 0)[0][1:]`).

In [ ]:
files, fig_meta = ajile.figshare_files()
print("Figshare item", fig_meta["id"], "version", fig_meta["version"], fig_meta["doi"])
proj, cents, link_rows = {}, {}, []
el_first = {s: [m for m in metas if int(m["subject_id"]) == s][0]["electrodes"] for s in subjects}
for s in subjects:
    m, c = ajile.load_projection(s, files)
    el = el_first[s]
    known = el.x.notna().values
    a, note = ajile.match_projection(m, known)
    # rule as the claims file words it: drop all-zero rows only
    allz = m.values.astype(float)
    n_nonzero = int((allz.mean(axis=1) != 0).sum())
    proj[s] = (list(m.columns), a)
    cents[s] = c
    link_rows.append(dict(subject=s, csv_rows=len(m), electrodes=len(el), with_position=int(known.sum()),
                          nonzero_rows=n_nonzero, code_rule=note, matched=a is not None))
link = pd.DataFrame(link_rows)
print(link.to_string())
cols_all = proj[1][0]
# The first non-zero row of each file, the row the code drops, holds the electrode density of each
# region (the remaining rows sum to 1 per region). Regions kept: mean density over subjects > 3 (Methods).
dens_rows = []
for s in subjects:
    m, _ = ajile.load_projection(s, files)
    a = m.values.astype(float)
    first = a[np.flatnonzero(a.mean(axis=1) != 0)[0]]
    rest = a[np.flatnonzero(a.mean(axis=1) != 0)[1:]]
    dens_rows.append(first)
    if s == 1:
        print("S01: rows after the first sum per region to (first 6 regions):", rest.sum(axis=0)[:6].round(3))
dens = np.array(dens_rows).mean(axis=0)
print("mean density of the eight regions:", {r: round(float(dens[cols_all.index(r)]), 2) for r in ajile.ROIS})
kept = [cols_all[i] for i in np.flatnonzero(dens > 3)]
print("regions with mean density > 3:", kept)
print("same as the eight of the claims file:", sorted(kept) == sorted(ajile.ROIS))

In [ ]:
# Largest-weight region against the nearest region centroid, with mirrored MNI positions
rng = np.random.default_rng(seed("roi-projection-link"))
def nearest_match(s, rng=None, n_shuffle=200):
    names, a = proj[s]
    c = cents[s]
    cn = [str(x) for x in c.columns]
    cent = c.values.astype(float)            # one row per AAL region, left-side centroids
    el = el_first[s]
    known = el.x.notna().values
    pos = el[["x", "y", "z"]].values[known].copy()
    pos[:, 0] = -np.abs(pos[:, 0])             # mirror to the left hemisphere
    W = a[known]
    has = W.sum(axis=1) > 0
    valid = np.isfinite(cent).all(axis=1) & (np.abs(cent).sum(axis=1) > 0)
    d = np.linalg.norm(pos[:, None, :] - cent[None, :, :], axis=2)
    d[:, ~valid] = np.inf
    near = d.argmin(axis=1)
    top = np.where(valid[None, :], W, -np.inf).argmax(axis=1)
    obs = int((near[has] == top[has]).sum())
    null = []
    for _ in range(n_shuffle):
        p = rng.permutation(len(W))
        topp = np.where(valid[None, :], W[p], -np.inf).argmax(axis=1)
        null.append(int((near[has] == topp[has]).sum()))
    return obs, int(has.sum()), np.array(null)
print("centroid table shape:", cents[1].shape, "columns:", list(cents[1].columns))
res_link = []
for s in subjects:
    obs, n, null = nearest_match(s, rng)
    res_link.append(dict(subject=s, matched_electrodes=n, top_is_nearest=obs, frac=obs / n,
                         shuffled_mean=null.mean(), shuffled_max=null.max(), beats_all_shuffles=obs > null.max()))
res_link = pd.DataFrame(res_link)
print(res_link.to_string())
# seed sensitivity
for off in SEED_OFFSETS[1:]:
    r2_ = np.random.default_rng(seed("roi-projection-link") + off)
    print("seed +", off, "subjects beating every shuffle:", sum(nearest_match(s, r2_)[0] > nearest_match(s, r2_)[2].max() for s in subjects))
n_ok = int((link.matched & res_link.beats_all_shuffles).sum())
print("subjects with rows = electrodes and positions fitting the weights:", n_ok, "of 12")

**Result: reproduced.** The paper's code rule (keep the non-zero rows, drop the first) leaves as
many rows as electrodes with a position in all 12 subjects; the dropped first row is the electrode
density of each region, and the regions whose mean density exceeds 3 are exactly the eight of the
claims file. For every subject the largest-weight region is the nearest region centroid for 36 to
55% of electrodes, against at most 14 electrodes in 200 shuffles of the electrode order, under all
five seeds. The 16 electrodes of S03 and the 8 of S12 without positions have no row; they get no
weight.

## Spectral power of every event (streamed)

For each file, the chunk index of the ECoG series is read from the file's B-tree, and the chunks
holding the 10 s around each event are fetched and decoded. The helper's docstring gives the source
of every parameter. Every event's segment is checked by the Methods' rule, and its power is
projected to the eight regions with the weights of that file's good electrodes.

In [ ]:
# SPECTRAL
t = time.time()
roi_idx = [cols_all.index(r) for r in ajile.ROIS]
jobs = []
for m in metas:
    s = int(m["subject_id"])
    good = m["electrodes"].good.values.astype(bool)
    w = ajile.session_weights(proj[s][1][:, roi_idx], good)
    jobs.append((m["asset_id"], m["event_times"], good, w))
spec = ajile.run_spectral(jobs, workers=8)
print(f"spectral pass {time.time() - t:.0f} s; peak memory of a worker "
      f"{resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.2f} GB")

In [ ]:
ev = ev_all.copy()
ev["inside"] = np.concatenate([r["inside"] for r in spec])
ev["allzero"] = np.concatenate([r["allzero"] for r in spec])
ev["seg_db"] = np.concatenate([r["seg_db"] for r in spec])
ev["hf_db"] = np.concatenate([r["hf_db"] for r in spec])
REG = np.concatenate([r["region"] for r in spec])          # (event, region, freq, bin), dB, no baseline
REG_BS = REG - REG[..., ajile.BASE].mean(axis=-1, keepdims=True)
ELEC = {k: [r["elec"][k] for r in spec] for k in spec[0]["elec"]}  # per file: (event, electrode)
hf_thr = ev[ev.inside & ev.hf_db.notna()].groupby("subject").hf_db.agg(lambda x: x.mean() + 3 * x.std())
ev["hf_bad"] = ev.hf_db > ev.subject.map(hf_thr)
ev["low_power"] = ev.seg_db < 0
# Segments holding NaN samples (the files store missing ECoG as NaN) have NaN power: missing data
ev["nan_data"] = ev.inside & ev.seg_db.isna()
ev["keep"] = ev.inside & ~ev.nan_data & ~ev.allzero & ~ev.low_power & ~ev.hf_bad
drop = ev.groupby("subject").agg(events=("keep", "size"), no_ecog=("inside", lambda x: int((~x).sum())),
                                 nan_data=("nan_data", "sum"), zeroed=("allzero", "sum"), below_0dB=("low_power", "sum"),
                                 hf_3sd=("hf_bad", "sum"), kept=("keep", "sum"))
print(drop.to_string())
print("segment power (dB, uV^2/Hz, mean 1-200 Hz): range", np.nanmin(ev.seg_db).round(2), "to", np.nanmax(ev.seg_db).round(2))
print("region power without baseline (dB): range", np.nanmin(REG).round(1), "to", np.nanmax(REG).round(1))
print("NaN region values among kept events:", int(np.isnan(REG[ev.keep.values]).sum()))

In [ ]:
# Clock check: event-locked band power at the true onsets against a pseudo-onset 3 s earlier in the
# same segments (same electrodes, same events). Precentral-weighted mean over good electrodes.
rows = []
for s in subjects:
    lf, hf, lfs, hfs = [], [], [], []
    for m, r in zip(metas, spec):
        if int(m["subject_id"]) != s:
            continue
        k = ev[(ev.asset_id == m["asset_id"])].keep.values
        good = m["electrodes"].good.values.astype(bool)
        w = ajile.session_weights(proj[s][1][:, roi_idx], good)[:, 1]
        w = np.nan_to_num(w)
        for key, lst in (("lfb_mean", lf), ("hfb_mean", hf), ("lfb_mean_shift", lfs), ("hfb_mean_shift", hfs)):
            lst.append(np.nansum(r["elec"][key][k] * w[None, :], axis=1))
    lf, hf, lfs, hfs = map(np.concatenate, (lf, hf, lfs, hfs))
    rows.append(dict(subject=s, lfb_onset=np.median(lf), lfb_pseudo=np.median(lfs),
                     hfb_onset=np.median(hf), hfb_pseudo=np.median(hfs),
                     p_hfb=stats.wilcoxon(hf, hfs).pvalue, p_lfb=stats.wilcoxon(lf, lfs).pvalue))
clock = pd.DataFrame(rows)
print(clock.round(4).to_string())
# Pose: contralateral wrist speed around onset
spd = []
for m, r in zip(metas, spec):
    w = "R_Wrist" if m["event_wrist"] == "r_wrist" else "L_Wrist"
    p = r["wrist"][w]
    spd.append(np.linalg.norm(np.diff(p, axis=1), axis=2) * 30)
spd = np.concatenate(spd)
tt = spec[0]["wrist_t"][1:]
print("median contralateral wrist speed (px/s) at -1.0, -0.5, 0, +0.25, +0.5 s:",
      [round(float(np.nanmedian(spd[:, np.argmin(np.abs(tt - x))])), 1) for x in (-1.0, -0.5, 0.0, 0.25, 0.5)])

In [ ]:
# Reach angle convention: the stored angle against the angle of the wrist's largest displacement
# from its onset position, computed from the pose (events up to 1.5 s long, the pose window)
W = np.concatenate([r["wrist"]["R_Wrist" if m["event_wrist"] == "r_wrist" else "L_Wrist"] for m, r in zip(metas, spec)])
img_ang, pmag = np.full(len(ev), np.nan), np.full(len(ev), np.nan)
for i in range(len(ev)):
    if ev.duration.values[i] > 1.5:
        continue
    n = int(ev.duration.values[i] * 30)
    seg = W[i, 45:46 + n] - W[i, 45]
    d = np.hypot(seg[:, 0], seg[:, 1])
    if np.isfinite(d).any():
        j = np.nanargmax(d)
        img_ang[i] = np.degrees(np.arctan2(seg[j, 1], seg[j, 0]))
        pmag[i] = d[j]
ok = np.isfinite(img_ang)
adiff = lambda a, b: np.abs((a - b + 180) % 360 - 180)
print("events compared:", ok.sum())
print("median |stored - image angle (y down)|:", np.median(adiff(ev.angle_raw.values[ok], img_ang[ok])).round(2), "deg")
print("median |stored - angle with y up|:", np.median(adiff(ev.angle_raw.values[ok], -img_ang[ok])).round(2), "deg")
print("median pose magnitude / stored magnitude:", np.median(pmag[ok] / ev.magnitude.values[ok]).round(3))

## Event sets

Reading (a): every event in the reaches tables. Reading (b): the events left after the Methods'
segment rejection (no ECoG for the whole 10 s segment, zeroed data, power below 0 dB, or 115-125 Hz
power more than 3 SD above the subject's segments). The dataset has no false-positive flag; the
Methods say those events were removed by visual inspection, so none are dropped for it here.

In [ ]:
EVA = ev
EVB = ev[ev.keep]
print("reading a:", len(EVA), "events; reading b:", len(EVB))

## subject-days

> The dataset's subjects and recording days are the paper's subjects S01 to S12 and the recording
> days the paper lists for each.

In [ ]:
paper_days = {1: [3, 4, 5, 7], 2: [3, 4, 5, 6], 3: [3, 4, 5, 6], 5: [3, 4, 7]}
paper_days = {s: paper_days.get(s, [3, 4, 5, 6, 7]) for s in range(1, 13)}
cmp = pd.DataFrame({"paper": pd.Series(paper_days).map(str), "dataset": days.map(lambda d: str([int(x) for x in d]))})
cmp["same"] = [paper_days[s] == [int(x) for x in days[s]] for s in cmp.index]
print(cmp.to_string())
print("subject-days in the dataset:", sum(len(d) for d in days), "; all match:", cmp.same.all())

**Result: reproduced.** The dataset's subjects 01 to 12 have exactly the days of Table 2, 55
subject-days in all.

## cohort

> Twelve subjects (8 male, 4 female, 29.4 +- 7.9 years) had on average 87.0 +- 12.9 surface
> electrodes; five also had 23.2 +- 12.1 depth electrodes; five were implanted in the right
> hemisphere and seven in the left.

Surface electrodes are those of groups whose description is not a depth electrode; depth electrodes
are those of groups described as depth. The hemisphere is the side of the mean x (MNI) of the
subject's electrodes with positions.

In [ ]:
rows = []
for s in subjects:
    m = [x for x in metas if int(x["subject_id"]) == s][0]
    el = m["electrodes"]
    gdesc = el.group_name.map(m["groups"]).fillna("")
    depth = gdesc.str.contains("depth", case=False)
    rows.append(dict(subject=s, sex=m["sex"], age=int(re.findall(r"\d+", m["age"])[0]), electrodes=len(el),
                     surface=int((~depth).sum()), depth=int(depth.sum()), mean_x=el.x.mean(),
                     hemisphere="right" if el.x.mean() > 0 else "left",
                     groups=", ".join(f"{g} ({m['groups'].get(g, '?')})" for g in el.group_name.unique())))
coh = pd.DataFrame(rows)
print(coh.drop(columns="groups").to_string())
for _, r in coh.iterrows():
    print(f"S{r.subject:02d}:", r.groups)
dsub = coh[coh.depth > 0]
vals = dict(subjects=len(coh), male=int((coh.sex == "M").sum()), mean_age=coh.age.mean(), sd_age=coh.age.std(),
            mean_surface=coh.surface.mean(), sd_surface=coh.surface.std(), with_depth=len(dsub),
            mean_depth=dsub.depth.mean(), sd_depth=dsub.depth.std(), right=int((coh.hemisphere == "right").sum()))
print({k: round(float(v), 2) for k, v in vals.items()})
exp = [("subjects", 12, 0), ("male", 8, 0), ("mean_age", 29.4, 7.9), ("mean_surface", 87.0, 12.9),
       ("with_depth", 5, 0), ("mean_depth", 23.2, 12.1), ("right", 5, 0)]
for k, v, tol in exp:
    print(f"{k}: measured {vals[k]:.2f}, expected {v} +- {tol}: {'within' if abs(vals[k] - v) <= tol else 'OUTSIDE'}")
# variant: the grid only as surface (a reading of 'surface' the data do not settle)

**Result: partially reproduced.** Twelve subjects, 8 male, mean age 29.42 years (SD 7.94), 84.83 surface
electrodes on average and 5 right-hemisphere implants match. Electrode groups described as depth
are in 7 subjects, not 5 (S01, S02, S03, S09, S10, S11, S12), with 23.71 depth electrodes on
average, within the stated SD of 23.2.

## fig2-event-counts

> Each subject contributed between 151 and 947 movement events (median 640 across subjects).

In [ ]:
cnt = pd.DataFrame({"a_all": EVA.groupby("subject").size(), "b_kept": EVB.groupby("subject").size(),
                    "with_ecog": ev[ev.inside & ~ev.nan_data].groupby("subject").size()})
print(cnt.to_string())
for col in cnt:
    c = cnt[col]
    ok = abs(c.min() - 151) <= 15.1 and abs(c.max() - 947) <= 94.7 and abs(c.median() - 640) <= 64
    print(f"{col}: min {c.min()}, max {c.max()}, median {c.median()} -> all three within tolerance: {ok}")

**Result: not reproduced.** Under no reading are all three values within tolerance. All events: 211
to 996, median 770.5. After the segment rejection: 210 to 976, median 691.0. Events with ECoG and
no missing samples: 211 to 986, median 700.5. The paper gives 151 to 947, median 640; the fewest
events are 210 or more in every reading, outside 151 +- 15.1.

## fig2-features-differ-across-subjects

> All ten behavioural and environmental features differ significantly across subjects.

In [ ]:
def kw_by_subject(E):
    out = {}
    for f in ajile.FEATURES:
        groups = [g[f].values for _, g in E.groupby("subject")]
        out[f] = stats.kruskal(*groups)
    return pd.DataFrame({f: dict(H=r.statistic, p=r.pvalue) for f, r in out.items()}).T
kwb = kw_by_subject(EVB); kwa = kw_by_subject(EVA)
tab = kwb.join(kwa, lsuffix="_b", rsuffix="_a")
print(tab.to_string(float_format=lambda x: f"{x:.3g}"))
print("reading b: p < 0.001 for all ten:", (kwb.p < 0.001).all(), "; reading a:", (kwa.p < 0.001).all())
print("largest p (b):", kwb.p.max(), kwb.p.idxmax())

**Result: reproduced.** All ten features differ across subjects (Kruskal-Wallis), with the largest p
1.1e-32 (reach duration) on the events of reading (b), and p < 0.001 for all ten on all events.

## fig2-reach-shape

> Reach duration and reach magnitude vary little across subjects, with most reaches lasting under
> 2 s and covering under 200 pixels, and reach angle distributions tend to be bimodal with peaks near
> +90 and -90 degrees, with few side-to-side reaches.

Angle density: Gaussian KDE with Scott's bandwidth on the folded angle, evaluated on -90 to 90 in
steps of 0.5 degrees (a standard grid). The grid's end points count as local maxima when they exceed
their neighbour, since the folded angle cannot pass +-90 (standard choice, stated).

In [ ]:
grid = np.arange(-90, 90.01, 0.5)
def bimodal(a):
    d = stats.gaussian_kde(a)(grid)
    peaks = [i for i in range(len(d)) if (i == 0 or d[i] > d[i - 1]) and (i == len(d) - 1 or d[i] > d[i + 1])]
    top = sorted(peaks, key=lambda i: -d[i])[:2]
    pk = sorted(grid[top])
    d0 = d[np.argmin(np.abs(grid))]
    ok = len(pk) == 2 and abs(pk[0] + 90) <= 15 and abs(pk[1] - 90) <= 15 and d0 < min(d[top])
    return ok, pk, d0, [d[i] for i in top]
rows = []
for s, g in list(EVB.groupby("subject")) + [("pooled", EVB)]:
    ok, pk, d0, dp = bimodal(g.angle.values)
    rows.append(dict(subject=s, n=len(g), frac_dur_lt2=(g.duration < 2).mean(), median_dur=g.duration.median(),
                     frac_mag_lt200=(g.magnitude < 200).mean(), median_mag=g.magnitude.median(),
                     peaks=np.round(pk, 1), dens0=round(d0, 5), dens_peaks=np.round(dp, 5), bimodal_pm90=ok,
                     frac_side=(np.abs(g.angle) < 30).mean()))
rs = pd.DataFrame(rows)
print(rs.to_string())
per = rs[rs.subject != "pooled"]; pool = rs[rs.subject == "pooled"].iloc[0]
dur_part = pool.frac_dur_lt2 > 0.5 and (per.frac_dur_lt2 > 0.5).sum() >= 10
mag_part = pool.frac_mag_lt200 > 0.5 and (per.frac_mag_lt200 > 0.5).sum() >= 10
ang_part = pool.bimodal_pm90 and per.bimodal_pm90.sum() > 6
print("duration part:", dur_part, (per.frac_dur_lt2 > 0.5).sum(), "of 12; magnitude part:", mag_part,
      (per.frac_mag_lt200 > 0.5).sum(), "of 12; angle part:", ang_part, per.bimodal_pm90.sum(), "of 12")
print("ranges: duration", EVB.duration.min().round(3), "to", EVB.duration.max().round(3), "s; magnitude",
      EVB.magnitude.min().round(1), "to", EVB.magnitude.max().round(1), "px")
# reading a as a variant
ra = [bimodal(g.angle.values)[0] for _, g in EVA.groupby("subject")]
print("reading a: bimodal subjects", sum(ra), "; pooled", bimodal(EVA.angle.values)[0],
      "; duration<2 s in", int((EVA.groupby('subject').duration.apply(lambda x: (x < 2).mean()) > 0.5).sum()),
      "subjects; magnitude<200 in", int((EVA.groupby('subject').magnitude.apply(lambda x: (x < 200).mean()) > 0.5).sum()))

In [ ]:
fig, axs = plt.subplots(3, 4, figsize=(12, 7), sharex=True)
for ax, (s, g) in zip(axs.ravel(), EVB.groupby("subject")):
    ax.plot(grid, stats.gaussian_kde(g.angle.values)(grid))
    ax.set_title(f"S{s:02d}", fontsize=9)
fig.supxlabel("folded reach angle (deg, +90 up)"); fig.supylabel("density"); plt.tight_layout(); plt.show()
print("all angles lie in -90..90:", EVB.angle.between(-90, 90).all())

**Result: partially reproduced.** Most reaches last under 2 s (88 to 95% per subject, 91.8% pooled)
and cover under 200 pixels (59 to 100% per subject, 91.3% pooled), in all 12 subjects. The angle
part fails: the density of the folded angle has its two highest maxima within 15 degrees of -90 and
+90 with a lower density at 0 in none of the 12 subjects and not pooled. Pooled, the maxima are at
-71.5 and +75 degrees, and the density at 0 (0.00407) is above the lower maximum (0.004). Reading
(a) gives the same.

## fig2-bimanual-skew

> Bimanual ratio and bimanual overlap are skewed toward unimanual movement of the contralateral arm,
> less so for subjects 02, 04 and 05, while the bimanual class is mostly bimanual.

In [ ]:
rows = []
for s, g in list(EVB.groupby("subject")) + [("pooled", EVB)]:
    rows.append(dict(subject=s, ratio_median=g.bimanual_ratio.median(), overlap_skew=stats.skew(g.bimanual_overlap),
                     overlap_median=g.bimanual_overlap.median(), class_frac_bimanual=g.bimanual_class.mean()))
bs_ = pd.DataFrame(rows)
print(bs_.to_string())
per = bs_[bs_.subject != "pooled"].set_index("subject"); pool = bs_[bs_.subject == "pooled"].iloc[0]
ratio_ok = pool.ratio_median < 0.5 and (per.ratio_median < 0.5).sum() >= 9
over_ok = pool.overlap_skew > 0 and (per.overlap_skew > 0).sum() >= 9
class_ok = pool.class_frac_bimanual > 0.5 and (per.class_frac_bimanual > 0.5).sum() > 6
top_ratio = per.ratio_median.sort_values(ascending=False).index[:5].tolist()
low_skew = per.overlap_skew.sort_values().index[:5].tolist()
named = [2, 4, 5]
less_ratio = all(s in top_ratio for s in named); less_over = all(s in low_skew for s in named)
print("ratio part:", ratio_ok, int((per.ratio_median < 0.5).sum()), "of 12; overlap part:", over_ok,
      int((per.overlap_skew > 0).sum()), "of 12; class part:", class_ok, int((per.class_frac_bimanual > 0.5).sum()), "of 12")
print("five highest median ratios:", top_ratio, "-> S02, S04, S05 among them:", less_ratio)
print("five lowest overlap skewness:", low_skew, "-> S02, S04, S05 among them:", less_over)
print("bimanual ratio range", EVB.bimanual_ratio.min(), EVB.bimanual_ratio.max(), "; overlap range",
      EVB.bimanual_overlap.min(), EVB.bimanual_overlap.max(), "(the description says a fraction of the contralateral movement)")
print("overlap values above 1:", int((EVB.bimanual_overlap > 1).sum()))

**Result: reproduced.** The median bimanual ratio is below 0.5 and the overlap skewness positive in
12 of 12 subjects and pooled (0.14 and 1.11); more than half of the events are bimanual in 12 of 12
(73% pooled). S02, S04 and S05 are among the five subjects with the highest median ratio and among
the five with the lowest overlap skewness.

## extfig2-1c-contralateral-larger

> Across all 12 subjects, the median displacement of the contralateral wrist after movement onset is
> substantially greater than that of the ipsilateral wrist.

Contralateral wrist: the wrist named in the ReachEvents description of each file. Pose windows
-1.5 to +1.5 s at 30 Hz; peak over 0 to 1.5 s of the median over events.

In [ ]:
tw = spec[0]["wrist_t"]
contra, ipsi = [], []
for m, r in zip(metas, spec):
    c = "R_Wrist" if m["event_wrist"] == "r_wrist" else "L_Wrist"
    i = "L_Wrist" if c == "R_Wrist" else "R_Wrist"
    contra.append(r["wrist"][c]); ipsi.append(r["wrist"][i])
contra, ipsi = np.concatenate(contra), np.concatenate(ipsi)
base = (tw >= -1.5) & (tw <= -1.0); post = (tw >= 0) & (tw <= 1.5); onset = np.argmin(np.abs(tw))
def disp(P, reading):
    ref = np.nanmean(P[:, base], axis=1) if reading == "a" else P[:, onset]
    return np.linalg.norm(P - ref[:, None, :], axis=2)
rows = []
kb = ev.keep.values
for reading in ("a", "b"):
    Dc, Di = disp(contra, reading), disp(ipsi, reading)
    for s in subjects:
        sel = kb & (ev.subject.values == s)
        mc, mi = np.nanmedian(Dc[sel], axis=0), np.nanmedian(Di[sel], axis=0)
        rows.append(dict(reading=reading, subject=s, n=int(sel.sum()), contra_peak=np.nanmax(mc[post]),
                         ipsi_peak=np.nanmax(mi[post]), ratio=np.nanmax(mc[post]) / np.nanmax(mi[post])))
dd = pd.DataFrame(rows)
dd["holds"] = dd.contra_peak > dd.ipsi_peak
print(dd.round(3).to_string())
print(dd.groupby("reading").holds.sum())
print("events with missing pose in the window:", int(np.isnan(contra[kb]).any(axis=(1, 2)).sum()), "of", int(kb.sum()))

**Result: reproduced.** In all 12 subjects, under both readings of the rest position, the peak of the
median contralateral displacement over 0 to 1.5 s exceeds that of the ipsilateral wrist; the ratio
is 1.06 to 42.0 (reading b, smallest for S05 and S04, which move both arms most).

## extfig2-2-feature-correlations

> Over all events, reach magnitude correlates with reach duration (r = 0.26) and onset speed
> (r = 0.56), reach duration with bimanual overlap (r = 0.48), and the bimanual features with one
> another (overlap and ratio 0.51, class and ratio 0.50, overlap and class 0.61).

In [ ]:
pairs = [("magnitude", "duration", 0.26), ("magnitude", "onset_speed", 0.56), ("duration", "bimanual_overlap", 0.48),
         ("bimanual_overlap", "bimanual_ratio", 0.51), ("bimanual_class", "bimanual_ratio", 0.50),
         ("bimanual_overlap", "bimanual_class", 0.61)]
rows = []
for a_, b_, v in pairs:
    rb = EVB[[a_, b_]].corr().iloc[0, 1]; ra_ = EVA[[a_, b_]].corr().iloc[0, 1]
    rows.append(dict(pair=f"{a_} vs {b_}", paper=v, r_b=rb, r_a=ra_, within_b=abs(rb - v) <= 0.1 * v + 1e-9,
                     within_a=abs(ra_ - v) <= 0.1 * v + 1e-9))
cr = pd.DataFrame(rows)
print(cr.round(3).to_string())
print("within tolerance, reading b:", int(cr.within_b.sum()), "of 6; reading a:", int(cr.within_a.sum()), "of 6")

**Result: partially reproduced.** Five of the six correlations are within 10% of the paper's under
both event sets. Reach magnitude and duration correlate at r = 0.229 (reading b), below 0.26 - 0.026.

## fig2-days-and-waking

> Reach counts were fairly even across recording days except for subjects 04, 05 and 09, each with
> one day holding most events, and events occurred mostly during waking hours.

In [ ]:
pc = EVB.groupby(["subject", "day"]).size().unstack(fill_value=0)
print(pc.to_string())
busiest = (pc.max(axis=1) / pc.sum(axis=1))
wake = EVB.groupby("subject").time_of_day.apply(lambda x: ((x >= 8) & (x < 24)).mean())
out = pd.DataFrame({"busiest_day_frac": busiest, "busiest_day": pc.idxmax(axis=1), "waking_frac": wake})
print(out.round(3).to_string())
over = set(out.index[out.busiest_day_frac > 0.5]); named = {4, 5, 9}
print("subjects whose busiest day holds > half:", sorted(over), "; named S04, S05, S09")
print("day part exact:", over == named, "; waking part (all subjects > half):", bool((wake > 0.5).all()))
print("time of day range (h):", EVB.time_of_day.min().round(2), "to", EVB.time_of_day.max().round(2))
print("events in 0-8 h by subject:", EVB[EVB.time_of_day < 8].groupby("subject").size().to_dict())
pa = EVA.groupby(["subject", "day"]).size().unstack(fill_value=0)
print("reading a, subjects whose busiest day holds > half:", sorted(pa.index[(pa.max(axis=1) / pa.sum(axis=1)) > 0.5]))

**Result: partially reproduced.** Events are mostly in waking hours: 64 to 95% of each subject's events
fall between 08:00 and 24:00. The busiest day holds more than half of the events for S05 (0.606) and
S09 (0.519) and for no other subject; for S04 it holds 0.400, so the day part does not hold.

## fig3-group-lfb-decrease-hfb-increase

> At the group level, low-frequency power decreases and high-frequency power increases at movement
> initiation in several cortical regions.

Per subject, the median over events (reading b) of baseline-subtracted region power; the mean over
the 12 subjects; the group mean over -1.5 to -1 s subtracted again. Masking as in the code's
`tfr_boot_sig_mask` as the claims file gives it: each subject's baseline bins are drawn at random
within frequency, the median over subjects taken, about 2000 values per frequency; two-sided
percentile p; Benjamini-Hochberg over the region's frequency and time bins (-1.5 to +1.0 s, the
bins computed here); non-significant values set to 0.

In [ ]:
kbv = ev.keep.values
subj_v = ev.subject.values
SUBJ_MED = np.stack([np.nanmedian(REG_BS[kbv & (subj_v == s)], axis=0) for s in subjects])  # (subj, roi, f, b)
GROUP = SUBJ_MED.mean(axis=0)
GROUP = GROUP - GROUP[..., ajile.BASE].mean(axis=-1, keepdims=True)
base_idx = np.flatnonzero(ajile.BASE)

def bh(p, q=0.05):
    p = p.ravel(); n = len(p); o = np.argsort(p)
    thr = q * np.arange(1, n + 1) / n
    ok = p[o] <= thr
    k = np.max(np.flatnonzero(ok)) + 1 if ok.any() else 0
    sig = np.zeros(n, bool); sig[o[:k]] = True
    return sig

def mask_group(rng, n_boot=2000):
    masked = np.zeros_like(GROUP)
    for r in range(GROUP.shape[0]):
        S = SUBJ_MED[:, r]                                   # (subj, f, b)
        null = np.empty((n_boot, S.shape[1]))
        for f in range(S.shape[1]):
            pick = rng.integers(0, len(base_idx), size=(n_boot, S.shape[0]))
            vals = S[np.arange(S.shape[0])[None, :], f, base_idx[pick]]
            null[:, f] = np.median(vals, axis=1)
        obs = GROUP[r]
        p = np.empty_like(obs)
        for f in range(obs.shape[0]):
            for b in range(obs.shape[1]):
                p[f, b] = min(1.0, 2 * min((null[:, f] >= obs[f, b]).mean(), (null[:, f] <= obs[f, b]).mean()))
        sig = bh(p).reshape(obs.shape)
        masked[r] = np.where(sig, obs, 0.0)
    return masked

LOW = (ajile.FREQS >= 4) & (ajile.FREQS <= 30); HIGH = (ajile.FREQS >= 50) & (ajile.FREQS <= 120)
b0 = np.argmin(np.abs(ajile.BIN_T))
def fig3_outcome(M):
    rows = []
    for r, name in enumerate(ajile.ROI_NAMES):
        rows.append(dict(region=name, low_a=M[r][LOW][:, ajile.WIN].mean(), high_a=M[r][HIGH][:, ajile.WIN].mean(),
                         low_b=M[r][LOW][:, b0].mean(), high_b=M[r][HIGH][:, b0].mean()))
    t = pd.DataFrame(rows).set_index("region")
    t["holds_a"] = (t.low_a < 0) & (t.high_a > 0); t["holds_b"] = (t.low_b < 0) & (t.high_b > 0)
    return t
MASKS = {off: mask_group(np.random.default_rng(seed("fig3-group-lfb-decrease-hfb-increase") + off)) for off in SEED_OFFSETS}
MASKED = MASKS[0]
t3 = fig3_outcome(MASKED)
print(t3.round(3).to_string())
named3 = ["precentral", "postcentral", "inferior parietal"]
print("named regions holding under a:", int(t3.loc[named3, "holds_a"].sum()), "; under b:", int(t3.loc[named3, "holds_b"].sum()))
print("unmasked group values, same summaries:")
print(fig3_outcome(GROUP).round(3).to_string())
for off in SEED_OFFSETS[1:]:
    tt3 = fig3_outcome(MASKS[off])
    print("seed +", off, ": named holding a/b:", int(tt3.loc[named3, "holds_a"].sum()), int(tt3.loc[named3, "holds_b"].sum()))
print("fraction of bins significant per region:", {n: round(float((MASKED[r] != 0).mean()), 3) for r, n in enumerate(ajile.ROI_NAMES)})

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(13, 6), sharex=True, sharey=True)
lim = np.abs(MASKED).max()
for r, ax in enumerate(axs.ravel()):
    im = ax.pcolormesh(ajile.BIN_T, ajile.FREQS, MASKED[r], cmap="RdBu_r", vmin=-lim, vmax=lim, shading="nearest")
    ax.set_title(ajile.ROI_NAMES[r], fontsize=9); ax.axvline(0, color="k", lw=0.5)
fig.colorbar(im, ax=axs, label="dB"); fig.supxlabel("time from onset (s)"); fig.supylabel("Hz"); plt.show()
print("colour range +-", round(float(lim), 3), "dB; masked values range", MASKED.min().round(3), "to", MASKED.max().round(3))

**Result: reproduced.** In precentral, postcentral and inferior parietal regions the masked group
power over 4-30 Hz is below 0 and over 50-120 Hz above 0, both over 0 to 0.5 s (for example
precentral -0.758 and 0.445 dB) and at the bin of 0 s, under all five seeds. Middle frontal and
supramarginal show the same pattern; the temporal regions do not.

## fig3-sensorimotor-largest

> Spectral power changes are largest in precentral, postcentral and inferior parietal regions, and
> the three temporal regions show minimal movement-related changes.

In [ ]:
temporal = ["superior temporal", "middle temporal", "inferior temporal"]
def largest(M):
    win_c = (ajile.BIN_T >= -0.5) & (ajile.BIN_T <= 0.5)
    t = pd.DataFrame({"a_all_0_0.5": [np.abs(M[r][:, ajile.WIN]).mean() for r in range(8)],
                      "b_low": [np.abs(M[r][LOW][:, ajile.WIN]).mean() for r in range(8)],
                      "b_high": [np.abs(M[r][HIGH][:, ajile.WIN]).mean() for r in range(8)],
                      "c_all_-0.5_0.5": [np.abs(M[r][:, win_c]).mean() for r in range(8)]}, index=ajile.ROI_NAMES)
    res = {}
    for col in ["a_all_0_0.5", "c_all_-0.5_0.5"]:
        top3 = t[col].sort_values(ascending=False).index[:3].tolist()
        res[col] = (sorted(top3) == sorted(named3), sum(n in top3 for n in named3),
                    all(t.loc[x, col] < t.loc[named3, col].min() for x in temporal), top3)
    # (b): a region ranks high when it is in the top three in both ranges
    tl = t.b_low.sort_values(ascending=False).index[:3].tolist(); th = t.b_high.sort_values(ascending=False).index[:3].tolist()
    both = [n for n in named3 if n in tl and n in th]
    tmp_ok = all(t.loc[x, "b_low"] < t.loc[named3, "b_low"].min() and t.loc[x, "b_high"] < t.loc[named3, "b_high"].min() for x in temporal)
    res["b"] = (len(both) == 3, len(both), tmp_ok, (tl, th))
    return t, res
t3l, r3l = largest(MASKED)
print(t3l.round(4).to_string())
for k, v in r3l.items():
    print(f"reading {k}: three named are the top three {v[0]}; named in top three {v[1]}; temporal below all three {v[2]}; top {v[3]}")
for off in SEED_OFFSETS[1:]:
    _, rr = largest(MASKS[off])
    print("seed +", off, {k: (v[0], v[1], v[2]) for k, v in rr.items()})

**Result: reproduced.** Under all three readings and all five seeds the three largest regions are
postcentral, precentral and inferior parietal, and each temporal region is below all three (for
reading a, 0.462, 0.447 and 0.296 dB against at most 0.032).

## fig3-lfb-more-widespread

> Low-frequency power decreases are more spatially widespread than high-frequency increases and are
> also present in supramarginal and superior temporal regions.

In [ ]:
def widespread(M):
    neg = np.array([(M[r][LOW][:, ajile.WIN] < 0).sum() for r in range(8)])
    pos = np.array([(M[r][HIGH][:, ajile.WIN] > 0).sum() for r in range(8)])
    a_ = ((neg > 0).sum(), (pos > 0).sum())
    b_ = (neg.sum() / LOW.sum(), pos.sum() / HIGH.sum())
    present = {n: int(neg[ajile.ROI_NAMES.index(n)]) for n in ["supramarginal", "superior temporal"]}
    return neg, pos, a_, b_, present
neg, pos, a_, b_, present = widespread(MASKED)
print(pd.DataFrame({"sig_negative_4_30Hz_bins": neg, "sig_positive_50_120Hz_bins": pos}, index=ajile.ROI_NAMES).to_string())
print("(a) regions with any: low", a_[0], "high", a_[1], "-> low larger:", a_[0] > a_[1])
print("(b) bins per frequency bin: low", round(b_[0], 3), "high", round(b_[1], 3), "-> low larger:", b_[0] > b_[1])
print("significant negative low-frequency bins in supramarginal / superior temporal:", present)
for off in SEED_OFFSETS[1:]:
    _, _, aa, bb, pp = widespread(MASKS[off])
    print("seed +", off, "a:", aa, "b:", np.round(bb, 3), pp)

**Result: blocked (methods underspecified).** The readings disagree. Reading (b), bins per frequency
bin, gives 34.2 significant negative low-frequency bins against 23.9 positive high-frequency bins,
and holds under all five seeds. Reading (a), regions with any significant bin, gives 7 against 7
(not larger), and 6 or 8 against 8 under the other seeds. Supramarginal (30 bins) and superior
temporal (23 bins) do have significant low-frequency decreases.

## fig4-precentral-subject-variability

> In the precentral region every subject shows the low/high-frequency pattern, but amplitudes and
> frequency bands vary; subjects 03, 06, 07, 08 and 11 have increases up to 120 Hz, subjects 09 and
> 12 mainly at 60-80 Hz, and subjects 04 and 08 have narrower low-frequency decreases.

In [ ]:
P = SUBJ_MED[:, 1].copy()                                     # precentral (subj, f, b), unmasked
P = P - P[..., ajile.BASE].mean(axis=-1, keepdims=True)
spec4 = P[..., ajile.WIN].mean(axis=-1)                        # (subj, f)
F = ajile.FREQS
tab4 = pd.DataFrame(spec4, index=[f"S{s:02d}" for s in subjects], columns=[int(f) for f in F])
print(tab4.round(2).to_string())
p1 = pd.DataFrame({"lfb_8_32": spec4[:, ajile.LFB].mean(1), "hfb_76_100": spec4[:, ajile.HFB].mean(1)}, index=tab4.index)
p1["holds"] = (p1.lfb_8_32 < 0) & (p1.hfb_76_100 > 0)
print(p1.round(3).to_string())
part1 = bool(p1.holds.all())
i116, i121 = list(F).index(116), list(F).index(121)
part2_rows = {s: (spec4[subjects.index(s), i116] > 0 and spec4[subjects.index(s), i121] > 0) for s in [3, 6, 7, 8, 11]}
part2 = all(part2_rows.values())
hi = F >= 56
p3 = {}
for s in [9, 12]:
    v = spec4[subjects.index(s)]
    fmax = F[hi][np.argmax(v[hi])]; top = v[hi].max()
    p3[s] = (fmax, round(float(top), 3), round(float(v[(F >= 101)].mean()), 3), 61 <= fmax <= 76 and v[F >= 101].mean() < top / 2)
part3 = all(x[3] for x in p3.values())
low = F <= 46
width = {s: int((spec4[i][low] < spec4[i][low].min() / 2).sum()) for i, s in enumerate(subjects)}
fewest = sorted(width, key=lambda s: width[s])[:3]
ties = sorted(width.values())[:4]
part4 = 4 in fewest and 8 in fewest
print("part 1 (every subject):", part1, "; part 2:", part2_rows, part2, "; part 3:", p3, part3)
print("part 4 widths (bins below half the minimum, 1-46 Hz):", width, "; three fewest:", fewest, "(four smallest widths:", ties, ")", part4)
print("parts holding:", sum([part1, part2, part3, part4]), "of 4")

**Result: partially reproduced.** Three of four parts hold. S03, S06, S07, S08 and S11 have positive
power at 116 and 121 Hz; S09 and S12 peak at 66 and 71 Hz with little power above 100 Hz; S04 and S08
are among the three subjects with the narrowest low-frequency decrease (2 bins, with S07). Part 1
fails in one subject: S09 has a mean of -0.03 dB over 76-100 Hz.

## Banded precentral power per event

Band mean (Methods) and band median (code) of baseline-subtracted precentral power over 0 to 0.5 s,
and the band mean without baseline subtraction (for extfig5-1 and the Discussion claim).

In [ ]:
pre = REG[:, 1]
pre_bs = REG_BS[:, 1]
st_bs = ajile.band_stats(pre_bs); st_raw = ajile.band_stats(pre)
for k, v in st_bs.items():
    ev["pre_" + k] = v
for k, v in st_raw.items():
    ev["pre_raw_" + k] = v
EVB = ev[ev.keep]
print(EVB[["pre_lfb_mean", "pre_hfb_mean", "pre_lfb_median", "pre_hfb_median", "pre_raw_lfb_mean", "pre_raw_hfb_mean"]].describe().round(3).to_string())

## fig5-band-power-differs-across-subjects

> Precentral LFB and HFB power in the 0.5 s after movement onset differs significantly across subjects.

In [ ]:
for band in ("lfb", "hfb"):
    for st in ("mean", "median"):
        g = [x[f"pre_{band}_{st}"].values for _, x in EVB.groupby("subject")]
        r = stats.kruskal(*g)
        print(f"{band.upper()} band {st}: H = {r.statistic:.1f}, p = {r.pvalue:.3g}, n = {len(EVB)} events in 12 subjects")

**Result: reproduced.** Precentral band power differs across the 12 subjects in both bands, for band
means and band medians (Kruskal-Wallis H = 750 to 1048, p below the smallest double).

## table1-subject-band-power-ci

> Each subject's precentral band power lies in the 95% bootstrap interval Table 1 gives; LFB power
> is below baseline and HFB power above it for most subjects.

In [ ]:
T1 = {1: ((-1.87, -1.29), (0.03, 0.23)), 2: ((-1.12, -0.31), (0.07, 0.64)), 3: ((-2.11, -1.73), (1.39, 1.67)),
      4: ((-0.17, 0.01), (0.32, 0.52)), 5: ((-0.7, -0.38), (0.2, 0.48)), 6: ((-1.63, -1.11), (2.02, 2.37)),
      7: ((-0.84, -0.51), (1.14, 1.34)), 8: ((-0.3, 0.04), (0.36, 0.63)), 9: ((-0.57, -0.08), (-0.11, 0.23)),
      10: ((-0.87, -0.35), (0.01, 0.3)), 11: ((-2.3, -1.86), (0.09, 0.34)), 12: ((-0.66, -0.44), (0.06, 0.2))}
rng = np.random.default_rng(seed("table1-subject-band-power-ci"))
rows = []
for s, g in EVB.groupby("subject"):
    for bi, band in enumerate(("lfb", "hfb")):
        lo, hi_ = T1[s][bi]
        row = dict(subject=s, band=band, paper_lo=lo, paper_hi=hi_)
        for bst in ("mean", "median"):
            x = g[f"pre_{band}_{bst}"].values
            for stat, fn in (("mean", np.mean), ("median", np.median)):
                v = fn(x)
                row[f"{stat}_of_band{bst}"] = v
                row[f"in_{stat}_of_band{bst}"] = lo <= v <= hi_
            boot = np.array([np.median(x[rng.integers(0, len(x), len(x))]) for _ in range(5000)])
            row[f"ci_median_band{bst}"] = np.round(np.percentile(boot, [2.5, 97.5]), 2)
        rows.append(row)
t1 = pd.DataFrame(rows)
print(t1.round(3).to_string())
for col in [c for c in t1.columns if c.startswith("in_")]:
    print(col, int(t1[col].sum()), "of 24")
print("LFB below 0 (median of band mean):", int((t1[t1.band == "lfb"].median_of_bandmean < 0).sum()), "of 12; HFB above 0:",
      int((t1[t1.band == "hfb"].median_of_bandmean > 0).sum()), "of 12")

**Result: partially reproduced.** Under the best reading (mean over events of the band mean), 21 of
24 subject and band values lie in Table 1's intervals; the other readings give 17 or 18. The three
outside are S03 HFB (1.235 against 1.39 to 1.67), S06 LFB (-1.036 against -1.63 to -1.11) and S06 HFB
(1.596 against 2.02 to 2.37). LFB power is below 0 and HFB power above 0 in all 12 subjects (median
of band means).

## fig5-recording-day-effects

> Precentral band power differs significantly across recording days in LFB for subjects 03, 05, 06,
> 07, 08 and 11, and in HFB for subjects 03, 04, 07, 08 and 10, despite baseline subtraction.

In [ ]:
paper_sig = {"lfb": {3, 5, 6, 7, 8, 11}, "hfb": {3, 4, 7, 8, 10}}
rows = []
for s, g in EVB.groupby("subject"):
    for band in ("lfb", "hfb"):
        row = dict(subject=s, band=band, paper_sig=s in paper_sig[band])
        for bst in ("mean", "median"):
            p = stats.kruskal(*[x[f"pre_{band}_{bst}"].values for _, x in g.groupby("day")]).pvalue
            row[f"p_band{bst}"] = p; row[f"agree_band{bst}"] = (p < 0.05) == (s in paper_sig[band])
        row["day_means"] = np.round(g.groupby("day")[f"pre_{band}_mean"].mean().values, 2)
        rows.append(row)
t5 = pd.DataFrame(rows)
print(t5.to_string(float_format=lambda x: f"{x:.3g}"))
print("pairs agreeing, band mean:", int(t5.agree_bandmean.sum()), "of 24; band median:", int(t5.agree_bandmedian.sum()), "of 24")

**Result: partially reproduced.** With band means, 20 of 24 subject and band pairs agree with the
paper (19 with band medians). The day effect is significant where the paper finds none for S01 LFB
(p = 0.044), S01 HFB (p = 0.029) and S06 HFB (p = 8.5e-11), and not significant for S10 HFB
(p = 0.10), where the paper finds one.

## extfig5-1-baseline-reduces-day-variability

> Subtracting the -1.5 to -1 s baseline reduces the variability of band power across recording
> days, though clear variability remains.

In [ ]:
rows = []
for s, g in EVB.groupby("subject"):
    for band in ("lfb", "hfb"):
        rows.append(dict(subject=s, band=band, sd_with=g.groupby("day")[f"pre_{band}_mean"].mean().std(),
                         sd_without=g.groupby("day")[f"pre_raw_{band}_mean"].mean().std()))
ta = pd.DataFrame(rows); ta["reduced"] = ta.sd_with < ta.sd_without
print("(a) precentral, per subject"); print(ta.round(3).to_string())
# (b) per good electrode (good in every file of the subject)
rows = []
for s in subjects:
    idx = [i for i, m in enumerate(metas) if int(m["subject_id"]) == s]
    good_all = np.all([metas[i]["electrodes"].good.values for i in idx], axis=0)
    for band in ("lfb", "hfb"):
        with_, without = [], []
        for i in idx:
            k = ev[ev.asset_id == metas[i]["asset_id"]].keep.values
            with_.append(np.nanmean(ELEC[f"{band}_mean"][i][k][:, good_all], axis=0))
            without.append(np.nanmean(ELEC[f"{band}_mean_raw"][i][k][:, good_all], axis=0))
        sw, so = np.std(with_, axis=0, ddof=1), np.std(without, axis=0, ddof=1)
        rows.append(dict(subject=s, band=band, electrodes=int(good_all.sum()), frac_reduced=np.mean(sw < so),
                         n_reduced=int((sw < so).sum()), median_sd_with=np.median(sw), median_sd_without=np.median(so)))
tb = pd.DataFrame(rows)
print("(b) per good electrode"); print(tb.round(3).to_string())
for band in ("lfb", "hfb"):
    a_frac = ta[ta.band == band].reduced.mean()
    b_frac = tb[tb.band == band].n_reduced.sum() / tb[tb.band == band].electrodes.sum()
    print(f"{band.upper()}: (a) reduced in {a_frac:.2f} of subjects; (b) reduced in {b_frac:.3f} of electrodes")
print("SD with baseline, range over subjects (dB):", ta.sd_with.min().round(3), "to", ta.sd_with.max().round(3))

**Result: reproduced.** Baseline subtraction lowers the SD of day means in 10 of 12 subjects for LFB
and 11 of 12 for HFB (reading a), and at 91% and 90% of electrodes (reading b). Variability remains:
day effects are significant in 13 subject and band pairs (fig5-recording-day-effects), with SDs of
up to 0.85 dB after subtraction.

## discussion-day-changes-small

> Day-to-day changes in baseline-subtracted spectral power are small (about 1-2 dB) compared with
> spectral power without baseline subtraction (about 10-50 dB).

In [ ]:
rows = []
for s, g in EVB.groupby("subject"):
    for band in ("lfb", "hfb"):
        dm = g.groupby("day")[f"pre_{band}_mean"].mean()
        rows.append(dict(subject=s, band=band, max_dev=(dm - g[f"pre_{band}_mean"].mean()).abs().max(),
                         day_range=dm.max() - dm.min(), level_uV=g[f"pre_raw_{band}_mean"].mean(),
                         level_V=g[f"pre_raw_{band}_mean"].mean() - 120))
td = pd.DataFrame(rows)
print(td.round(3).to_string())
p1_ = bool((td.max_dev <= 2.2).all())
lev_uv = bool(td.level_uV.between(9, 55).all()); lev_v = bool(td.level_V.between(9, 55).all())
print("largest day deviation:", td.max_dev.max().round(3), "dB -> part 1:", p1_)
print("levels (uV^2 reading):", td.level_uV.min().round(2), "to", td.level_uV.max().round(2), "dB, in 9-55:", lev_uv,
      "; (V^2 reading):", td.level_V.min().round(2), "to", td.level_V.max().round(2), "in 9-55:", lev_v)
print("fraction of subject-band levels in 9-55 dB (uV^2):", td.level_uV.between(9, 55).mean().round(3))

**Result: reproduced.** The largest deviation of a day mean from the subject mean is 1.48 dB (S06 HFB),
at most 2.2 dB in every subject and band. Without baseline subtraction the precentral band power is
12.2 to 36.5 dB in microvolts squared, within 9 to 55 dB; in volts squared it would be about -108 to
-84 dB.

## Regression models (Figure 6)

One model per subject, electrode and band, on the events of reading (b). Electrodes are those good
in every file of the subject (the Methods remove bad electrodes; the flags differ between files, see
the data check). Ten features as the Methods and code encode them; forward selection over single
columns on all events (code); Huber regression (sklearn defaults); 200 day-balanced splits with 10%
test events (Methods), and 20% (code) as a variant. Delta R2 is computed where the model's own R2,
or its electrode's LFB R2, exceeds 0.05. The splits use the generator of claim
fig6b-best-models-sensorimotor, the 80/20 splits its seed + 1 and the shuffles its seed + 2; the
other Figure 6 claims use these same fits.

In [ ]:
t = time.time()
REG_SEED = seed("fig6b-best-models-sensorimotor")
def regression_jobs(seed0, include_bad=False):
    jobs = []
    for s in subjects:
        idx = [i for i, m in enumerate(metas) if int(m["subject_id"]) == s]
        E = pd.concat([ev[ev.asset_id == metas[i]["asset_id"]] for i in idx])
        keep = E.keep.values
        G = np.array([metas[i]["electrodes"].good.values for i in idx])
        use = np.ones(G.shape[1], bool) if include_bad else G.all(axis=0)
        Y = np.stack([np.concatenate([ELEC[k][i] for i in idx])[keep][:, use]
                      for k in ("lfb_mean", "hfb_mean", "lfb_median", "hfb_median")], axis=-1)
        finite = np.isfinite(Y).all(axis=(0, 2))
        Ek = E[keep]
        X, names = ajile.design_matrix(Ek)
        jobs.append(dict(subject=s, X=X, names=names, Y=Y[:, finite], days=Ek.day.values, seed=seed0,
                         electrodes=np.flatnonzero(use)[finite], targets=["lfb_mean", "hfb_mean", "lfb_median", "hfb_median"],
                         main_targets=["lfb_mean", "hfb_mean"], delta_min_r2=0.05, dropped_nonfinite=int((~finite).sum()),
                         good_all=G.all(axis=0)))
    return jobs
JOBS = regression_jobs(REG_SEED)
print("models per subject (electrodes):", {j["subject"]: j["Y"].shape[1] for j in JOBS},
      "; electrodes dropped for non-finite power:", {j["subject"]: j["dropped_nonfinite"] for j in JOBS})
REGR = pd.DataFrame(ajile.run_regression(JOBS, workers=4))
print(f"regression {time.time() - t:.0f} s, {len(REGR)} models")
REGR["band"] = REGR.target.str[:3]
MAIN = REGR[REGR.target.isin(["lfb_mean", "hfb_mean"])].copy()
print(MAIN.groupby("band").r2.describe().round(4).to_string())
print("Huber fits that did not converge (main models, 200 fits each):", int(MAIN.nonconverged.sum()), "of", 200 * len(MAIN),
      "; models with any:", int((MAIN.nonconverged > 0).sum()))

## fig6-r2-at-most-025

> All regression models explain at most 25% of the variance of withheld single-event power.

In [ ]:
mx = MAIN.r2.max(); row = MAIN.loc[MAIN.r2.idxmax()]
print(f"largest split-averaged withheld R2 (90/10, band mean): {mx:.4f} (S{row.subject:02d}, electrode {row.electrode}, {row.band})")
print("largest with 80/20 splits:", MAIN.r2_80.max().round(4), "; with band median:", REGR[REGR.target.str.endswith("median")].r2.max().round(4))
print("models with R2 > 0.05:", MAIN.groupby("band").r2.apply(lambda x: int((x > 0.05).sum())).to_dict(), "of", MAIN.groupby("band").size().to_dict())
print("ten largest:"); print(MAIN.nlargest(10, "r2")[["subject", "electrode", "band", "r2", "r2_80"]].round(4).to_string())

**Result: reproduced.** The largest split-averaged withheld R2 is 0.177 (S07, HFB), at most 0.275. It is
0.184 with 80/20 splits and 0.158 with band medians, and 0.165 to 0.184 under the other four seeds.

## fig6b-best-models-sensorimotor

> Regression models with the highest withheld R2 are mainly at electrodes over frontoparietal
> sensorimotor areas, for both bands.

In [ ]:
frontopar = ["middle frontal", "precentral", "postcentral", "inferior parietal", "supramarginal"]
sensor3 = ["precentral", "postcentral", "inferior parietal"]
def region_of(s, e):
    w = proj[s][1][e, roi_idx]
    return ajile.ROI_NAMES[int(np.argmax(w))] if w.sum() > 0 else "none"
MAIN["region"] = [region_of(s, e) for s, e in zip(MAIN.subject, MAIN.electrode)]
for band in ("lfb", "hfb"):
    M = MAIN[MAIN.band == band]
    sel = M[M.r2 > 0.05]
    frac = sel.region.isin(frontopar).mean() if len(sel) else np.nan
    med_s = M[M.region.isin(sensor3)].r2.median(); med_t = M[M.region.isin(temporal)].r2.median()
    # (b) projected R2: per subject weighted mean with the good-electrode weights, then mean over subjects
    proj_r2 = []
    for s in subjects:
        Ms = M[M.subject == s]
        w = np.nan_to_num(proj[s][1][Ms.electrode.values][:, roi_idx])
        w = w / np.where(w.sum(0) > 0, w.sum(0), np.nan)
        proj_r2.append(np.nansum(w * Ms.r2.values[:, None], axis=0))
    pr = pd.Series(np.nanmean(proj_r2, axis=0), index=ajile.ROI_NAMES).sort_values(ascending=False)
    print(f"{band.upper()}: (a) {len(sel)} models with R2 > 0.05, fraction in the five frontoparietal regions {frac:.3f};",
          f"median R2 sensorimotor {med_s:.4f} vs temporal {med_t:.4f}")
    print("   regions of the models with R2 > 0.05:", sel.region.value_counts().to_dict())
    print("   (b) projected R2 by region:", pr.round(4).to_dict(), "-> top three frontoparietal:", all(x in frontopar for x in pr.index[:3]))
    print("   models with R2 > 0.05 per subject:", sel.subject.value_counts().sort_index().to_dict())

**Result: reproduced.** Of the models with R2 > 0.05 (14 per band), all LFB and 13 of 14 HFB electrodes
are in the five frontoparietal regions, and the median R2 of sensorimotor electrodes is above that
of temporal ones in both bands. Projected onto the regions, the three highest are precentral,
postcentral and middle frontal in both bands.

## fig6c-angle-and-day-most-informative

> Reach angle and day of recording are the most informative features for single-event power,
> especially for low-frequency power.

In [ ]:
def delta_table(df, col="delta", select="own"):
    out = {}
    for band in ("lfb", "hfb"):
        M = df[df.band == band]
        if select == "own":
            M = M[M.r2 > 0.05]
        else:
            lfb_r2 = df[df.band == "lfb"].set_index(["subject", "electrode"]).r2
            M = M[[lfb_r2.get((s, e), -1) > 0.05 for s, e in zip(M.subject, M.electrode)]]
        D = pd.DataFrame([d for d in M[col] if isinstance(d, dict)])
        out[band] = D.median().sort_values(ascending=False) if len(D) else pd.Series(dtype=float)
        out[band + "_n"] = len(D)
    return out
for label, col, selm in (("Methods Delta R2, own-band selection", "delta", "own"),
                         ("code Delta R2 (80/20, test shuffled), own-band selection", "delta_code", "own"),
                         ("Methods Delta R2, selection by LFB R2 (code)", "delta", "lfb")):
    d = delta_table(MAIN, col, selm)
    print(label)
    for band in ("lfb", "hfb"):
        print(f"  {band.upper()} ({d[band + '_n']} models):", d[band].round(5).to_dict())
        print("     top two:", list(d[band].index[:2]))

**Result: reproduced.** Over the 14 LFB models with R2 > 0.05, recording day (median Delta R2 0.038) and
reach angle (0.032) are the two most informative features; the other eight have medians of 0. The
same two lead in HFB, and under the code's Delta R2 and the code's selection by LFB R2.

## extfig6-2-angle-most-retained

> Reach angle is the feature most often retained by forward selection in sensorimotor regions,
> followed by reach duration, reach magnitude, bimanual ratio and bimanual overlap.

In [ ]:
RET = {}
for band in ("lfb", "hfb"):
    M = MAIN[MAIN.band == band]
    per_s = []
    for s in subjects:
        Ms = M[M.subject == s]
        R = pd.DataFrame(list(Ms.retention)).reindex(columns=ajile.FEATURES).values
        w = np.nan_to_num(proj[s][1][Ms.electrode.values][:, roi_idx])
        w = w / np.where(w.sum(0) > 0, w.sum(0), np.nan)
        per_s.append(np.einsum("er,ef->rf", np.nan_to_num(w), R))
    RET[band] = pd.DataFrame(np.nanmean(per_s, axis=0), index=ajile.ROI_NAMES, columns=ajile.FEATURES)
    print(band.upper()); print(RET[band].round(3).to_string())
angle_top = {(b, r): RET[b].loc[r].idxmax() for b in ("lfb", "hfb") for r in sensor3}
print("most retained feature in each sensorimotor region and band:", angle_top)
angle_part = all(v == "angle" for v in angle_top.values())
order_paper = ["duration", "magnitude", "bimanual_ratio", "bimanual_overlap"]
ra_ = pd.concat([RET[b].loc[sensor3] for b in RET]).mean().sort_values(ascending=False)
rb_ = pd.concat([RET[b] for b in RET]).mean().sort_values(ascending=False)
print("(a) three sensorimotor regions, both bands:", ra_.round(3).to_dict())
print("(b) all eight regions, both bands:", rb_.round(3).to_dict())
print("angle part:", angle_part, "; order part (a):", list(ra_.index[1:5]) == order_paper, list(ra_.index[1:5]),
      "; (b):", list(rb_.index[1:5]) == order_paper, list(rb_.index[1:5]))

**Result: partially reproduced.** Reach angle is the most often retained feature in precentral,
postcentral and inferior parietal regions for both bands (0.52 averaged over these). The order of the
next four does not hold: over the sensorimotor regions it is speech ratio, duration, magnitude and
onset speed, and over all regions speech ratio, magnitude, bimanual class and onset speed.

## fig6d-upward-reaches-enhance-pattern

> Upward reaches are associated with lower low-frequency power and higher high-frequency power.

In [ ]:
for label, selm in (("own-band R2 > 0.05", "own"), ("LFB R2 > 0.05 (code)", "lfb")):
    out = {}
    lfb_r2 = MAIN[MAIN.band == "lfb"].set_index(["subject", "electrode"]).r2
    for band in ("lfb", "hfb"):
        M = MAIN[MAIN.band == band]
        M = M[M.r2 > 0.05] if selm == "own" else M[[lfb_r2.get((s, e), -1) > 0.05 for s, e in zip(M.subject, M.electrode)]]
        c = np.array([d["angle"] for d in M.coef])
        out[band] = (len(c), np.median(c) if len(c) else np.nan, (c < 0).mean() if len(c) else np.nan)
    print(label, {b: dict(n=v[0], median_angle_coef=round(float(v[1]), 4), frac_negative=round(float(v[2]), 3)) for b, v in out.items()})

**Result: reproduced.** Over models with R2 > 0.05 the median reach angle coefficient is -0.70 for LFB
(13 of 14 negative) and +0.54 for HFB (13 of 14 positive); with the code's selection by LFB R2 the
HFB median is +0.24.

## fig6d-angle-and-day-largest-coefficients

> Reach angle and day of recording have the largest coefficient magnitudes among the features, and
> the SD of the recording-day coefficients is large.

In [ ]:
for label, selm in (("own-band R2 > 0.05", "own"), ("LFB R2 > 0.05 (code)", "lfb")):
    lfb_r2 = MAIN[MAIN.band == "lfb"].set_index(["subject", "electrode"]).r2
    for band in ("lfb", "hfb"):
        M = MAIN[MAIN.band == band]
        M = M[M.r2 > 0.05] if selm == "own" else M[[lfb_r2.get((s, e), -1) > 0.05 for s, e in zip(M.subject, M.electrode)]]
        C = pd.DataFrame(list(M.coef))
        ra = C.abs().median().sort_values(ascending=False); rb = C.median().abs().sort_values(ascending=False)
        print(f"{label}, {band.upper()} ({len(C)} models)")
        print("   (a) median |coef|:", ra.round(4).to_dict(), "top two", list(ra.index[:2]))
        print("   (b) |median coef|:", rb.round(4).to_dict(), "top two", list(rb.index[:2]))

**Result: reproduced.** Under both readings, recording day (SD of its one-hot coefficients) and reach
angle have the two largest coefficient magnitudes in both bands: LFB 0.98 and 0.70, HFB 0.71 and 0.54.
The code's selection by LFB R2 gives the same two.

In [ ]:
# Seed sensitivity of the regression: full-model R2 of the models with R2 > 0.03 under the main seed,
# refit with the splits of the claim's seed + 1000 ... + 4000 (Delta R2 not refit: see the summary)
t = time.time()
top = MAIN[MAIN.r2 > 0.03][["subject", "electrode"]].drop_duplicates()
sens = {}
for off in SEED_OFFSETS[1:]:
    jobs = []
    for j in JOBS:
        es = set(top[top.subject == j["subject"]].electrode)
        k = np.array([e in es for e in j["electrodes"]])
        if not k.any():
            continue
        jj = dict(j); jj["Y"] = j["Y"][:, k][:, :, :2]; jj["electrodes"] = j["electrodes"][k]
        jj["targets"] = ["lfb_mean", "hfb_mean"]; jj["seed"] = REG_SEED + off; jj["delta_min_r2"] = np.inf
        jobs.append(jj)
    R = pd.DataFrame(ajile.run_regression(jobs, workers=4))
    R["band"] = R.target.str[:3]
    sens[off] = R
    print(f"seed + {off}: largest R2 {R.r2.max():.4f}; models > 0.05:", R.groupby('band').r2.apply(lambda x: int((x > 0.05).sum())).to_dict())
print(f"{time.time() - t:.0f} s")

## Summary

| Claim | Outcome |
|---|---|
| subject-days | reproduced |
| roi-projection-link | reproduced |
| cohort | partially reproduced (depth electrodes in 7 subjects, not 5) |
| fig2-event-counts | not reproduced (210 to 976 events per subject, median 691) |
| fig2-features-differ-across-subjects | reproduced |
| fig2-reach-shape | partially reproduced (no angle density peaks within 15 degrees of +-90) |
| fig2-bimanual-skew | reproduced |
| extfig2-1c-contralateral-larger | reproduced |
| extfig2-2-feature-correlations | partially reproduced (5 of 6) |
| fig2-days-and-waking | partially reproduced (S04's busiest day holds 40%) |
| fig3-group-lfb-decrease-hfb-increase | reproduced |
| fig3-sensorimotor-largest | reproduced |
| fig3-lfb-more-widespread | blocked, methods underspecified (the two readings disagree) |
| fig4-precentral-subject-variability | partially reproduced (3 of 4 parts; S09 has no HFB increase) |
| fig5-band-power-differs-across-subjects | reproduced |
| table1-subject-band-power-ci | partially reproduced (21 of 24) |
| fig5-recording-day-effects | partially reproduced (20 of 24) |
| extfig5-1-baseline-reduces-day-variability | reproduced |
| discussion-day-changes-small | reproduced |
| fig6b-best-models-sensorimotor | reproduced |
| fig6-r2-at-most-025 | reproduced |
| fig6c-angle-and-day-most-informative | reproduced |
| extfig6-2-angle-most-retained | partially reproduced (order of the next four features differs) |
| fig6d-upward-reaches-enhance-pattern | reproduced |
| fig6d-angle-and-day-largest-coefficients | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper started from its own curated epochs and
spectral power (Figshare); here everything is computed from the dataset's continuous ECoG, with the
Methods' spectral method and the code's parameters. The segment rejection pools power over the good
electrodes and 1-200 Hz (an assumption). The good-electrode flags differ between a subject's files;
the projection uses each file's flags, and the regression uses electrodes good in every file. The
dataset has no false-positive flag, so none are removed beyond what it holds. The regression's
Delta R2 is computed only where R2 > 0.05 (the only models the claims use), the code's Delta R2
variant is run with the code's 80/20 splits, and the seed check of the regression refits the
full-model R2 only. Bad electrodes are not refit as a variant.

What was found about the dataset (numbers above):

- The subjects and days are those of the paper, and the paper's projection matrices match the
  electrodes in all 12 subjects. The event features, the pose and the ECoG agree with one another:
  the stored reach magnitude equals the one computed from the pose (ratio 1.007), and event-locked
  power changes at the true onsets differ from those at pseudo-onsets 3 s earlier (Wilcoxon p < 0.05)
  in 11 of 12 subjects for LFB and 9 of 12 for HFB. For S09 neither band differs, so the neural data
  could not establish the alignment there.
- The ECoG of three files ends hours before the day ends (S01 day 7, S04 day 3, S06 day 3), while the
  pose and the events go on: 237 events have no ECoG. Another 125 events fall on NaN samples.
- The dataset holds more events than the paper analysed (8088; 211 to 996 per subject, median 770.5,
  against 151 to 947, median 640).
- Undocumented conventions: the reach angle is in image coordinates (y down); the bimanual overlap,
  described as a fraction of the movement, is in seconds (1661 events above 1); the pose is labelled
  meters but is in pixels; good-electrode flags differ between a subject's files; 24 electrodes
  (S03, S12) have no position; the electrodes table carries R2 columns without a description.
- One event (S08 day 4) is at 40.3 s in ReachEvents and at 86440.3 s in the reaches table.
- Electrode groups described as depth are in 7 subjects, where the paper reports 5.

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")